# 00 — Setup (Google Colab)

Mount Google Drive, cài dependencies, cấu hình paths cho Colab Free.

**Cấu trúc Drive kỳ vọng**:
```
MyDrive/
  GenAI_ViMD/
    ViMD/
      data/          ← parquet files ở đây
        *.parquet
        __init__.py
      .cache/
      file/
      README.md
    storage/         ← tạo tự động
```

**Thứ tự chạy**: 00 → 01 → 03 → 04 → 05 → 06

In [ ]:
# ── Cell 0: Mount Google Drive ───────────────────────────────────────────────
import os, sys

IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
print(f'Running on: {"Google Colab" if IS_COLAB else "Local"}')

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('✓ Drive mounted at /content/drive')

    DRIVE_ROOT     = '/content/drive/MyDrive/GenAI_ViMD'
    PROJECT_ROOT   = DRIVE_ROOT
    # ViMD: parquet nằm trong ViMD/data/
    VIMD_PATH      = os.path.join(DRIVE_ROOT, 'ViMD', 'data')
    PROCESSED_PATH = os.path.join(DRIVE_ROOT, 'storage', 'processed')
    DATASET_EXPORT = os.path.join(DRIVE_ROOT, 'storage', 'datasets', 'hf_dataset')
    WEIGHTS_DIR    = os.path.join(DRIVE_ROOT, 'storage', 'weights')
    CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, 'storage', 'checkpoints')
    LOGS_DIR       = os.path.join(DRIVE_ROOT, 'storage', 'logs')
    EXPORTS_DIR    = os.path.join(DRIVE_ROOT, 'storage', 'exports')
else:
    from pathlib import Path
    _nb_dir = Path(os.path.abspath('.')).resolve()
    if _nb_dir.name == 'notebooks':
        PROJECT_ROOT = str(_nb_dir.parent.parent)
    else:
        PROJECT_ROOT = str(_nb_dir)
    VIMD_PATH      = os.path.join(PROJECT_ROOT, 'ViMD', 'data')
    PROCESSED_PATH = os.path.join(PROJECT_ROOT, 'storage', 'processed')
    DATASET_EXPORT = os.path.join(PROJECT_ROOT, 'storage', 'datasets', 'hf_dataset')
    WEIGHTS_DIR    = os.path.join(PROJECT_ROOT, 'storage', 'weights')
    CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'storage', 'checkpoints')
    LOGS_DIR       = os.path.join(PROJECT_ROOT, 'storage', 'logs')
    EXPORTS_DIR    = os.path.join(PROJECT_ROOT, 'storage', 'exports')

# Export env vars
for k, v in [
    ('PROJECT_ROOT',   PROJECT_ROOT),
    ('VIMD_PATH',      VIMD_PATH),
    ('PROCESSED_PATH', PROCESSED_PATH),
    ('DATASET_EXPORT', DATASET_EXPORT),
    ('WEIGHTS_DIR',    WEIGHTS_DIR),
    ('CHECKPOINT_DIR', CHECKPOINT_DIR),
    ('LOGS_DIR',       LOGS_DIR),
    ('IS_COLAB',       str(IS_COLAB)),
]:
    os.environ[k] = v

print(f'PROJECT_ROOT   : {PROJECT_ROOT}')
print(f'VIMD_PATH      : {VIMD_PATH}')
print(f'DATASET_EXPORT : {DATASET_EXPORT}')
print(f'CHECKPOINT_DIR : {CHECKPOINT_DIR}')

In [ ]:
# ── Cell 1: Kiểm tra ViMD data ─────────────────────────────────────────────
import glob

# ViMD/data/ chứa parquet files
if not os.path.isdir(VIMD_PATH):
    print(f'❌ ViMD/data folder không tìm thấy: {VIMD_PATH}')
    print()
    # Kiểm tra ViMD root có tồn tại không
    vimd_root = os.path.join(os.path.dirname(VIMD_PATH), '..')
    vimd_parent = os.path.dirname(VIMD_PATH)
    if os.path.isdir(vimd_parent):
        print(f'ViMD root ({vimd_parent}) exists. Contents:')
        print(os.listdir(vimd_parent))
    else:
        print(f'Hãy kiểm tra: ls /content/drive/MyDrive/GenAI_ViMD/ViMD/')
else:
    # Tìm tất cả parquet trong ViMD/data/
    parquet_files = glob.glob(os.path.join(VIMD_PATH, '*.parquet'))
    # Thêm recursive scan đề phòng có subfolder
    parquet_files += glob.glob(os.path.join(VIMD_PATH, '**', '*.parquet'), recursive=True)
    parquet_files = sorted(set(parquet_files))

    if parquet_files:
        print(f'✓ ViMD/data: {len(parquet_files)} parquet shards')
        print(f'  Sample: {[os.path.basename(f) for f in parquet_files[:3]]}')
    else:
        print(f'⚠️  Không tìm thấy .parquet trong {VIMD_PATH}')
        print(f'  Nội dung: {os.listdir(VIMD_PATH)[:15]}')

In [ ]:
# ── Cell 2: Tạo thư mục cần thiết ─────────────────────────────────────────
from pathlib import Path

dirs_to_create = [
    PROCESSED_PATH,
    os.path.join(PROCESSED_PATH, 'features'),
    DATASET_EXPORT,
    WEIGHTS_DIR,
    os.path.join(WEIGHTS_DIR, 'dialect'),
    os.path.join(WEIGHTS_DIR, 'asr_wav2vec2'),
    os.path.join(WEIGHTS_DIR, 'translation'),
    os.path.join(WEIGHTS_DIR, 'xtts-v2'),
    os.path.join(CHECKPOINT_DIR, 'asr_wav2vec2'),
    os.path.join(CHECKPOINT_DIR, 'translation'),
    os.path.join(CHECKPOINT_DIR, 'tts'),
    LOGS_DIR,
    os.path.join(LOGS_DIR, 'mlflow'),
    EXPORTS_DIR,
]

for d in dirs_to_create:
    Path(d).mkdir(parents=True, exist_ok=True)

print('✓ Tất cả thư mục đã được tạo')
print(f'  storage/ → {os.path.join(PROJECT_ROOT, "storage")}')

In [ ]:
# ── Cell 3: Cài dependencies (Colab Free) ──────────────────────────────────
# Colab Free: cài mỗi session (không persist qua restart)
# Tránh cài package nặng không cần thiết để tiết kiệm RAM (~12GB trên T4)
import subprocess, sys

def pip_install(*packages, quiet=True):
    q = ['-q'] if quiet else []
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install'] + q + list(packages),
        capture_output=True, text=True
    )
    return r.returncode == 0

print('Cài core ML...')
pip_install('--upgrade', 'pip')
pip_install('tqdm', 'numpy', 'pandas', 'pyarrow')

print('Cài HuggingFace...')
pip_install('transformers>=4.40.0', 'datasets>=2.18.0',
            'accelerate>=0.28.0', 'evaluate>=0.4.0', 'jiwer>=3.0.0')

print('Cài audio...')
pip_install('librosa>=0.10.0', 'soundfile', 'noisereduce', 'audiomentations')

print('Cài speech/ASR...')
pip_install('speechbrain>=1.0.0', 'faster-whisper')

print('Cài utilities...')
pip_install('mlflow>=2.10.0', 'sacrebleu', 'sentencepiece', 'scikit-learn', 'psutil')

# TTS optional (nặng, skip nếu không cần 05/06)
print('Cài TTS (optional — có thể chậm)...')
ok = pip_install('TTS')
print(f'  TTS: {"✓" if ok else "⚠️  skip"}')

print('\n✓ Dependencies installed')

In [ ]:
# ── Cell 4: Kiểm tra GPU (Colab Free = T4 15GB) ───────────────────────────
print('=' * 55)
print('Colab Free Environment Check')
print('=' * 55)
print(f'Python: {sys.version.split(" ")[0]}')

try:
    import torch
    print(f'PyTorch: {torch.__version__}')
    print(f'CUDA   : {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'GPU    : {gpu_name}')
        print(f'VRAM   : {vram_gb:.1f} GB')
        if 'T4' in gpu_name:
            print('✓ T4 GPU — batch_size=8 OK cho wav2vec2-base')
        elif vram_gb >= 15:
            print('✓ VRAM đủ cho wav2vec2-base')
        else:
            print('⚠️  VRAM < 15GB — dùng batch_size=4')
    else:
        print('⚠️  Không có GPU!')
        print('   Colab: Runtime → Change runtime type → T4 GPU')
except ImportError:
    print('❌ PyTorch chưa cài — chạy lại Cell 3')

# Kiểm tra RAM (Colab Free ~12.7GB)
try:
    import psutil
    ram = psutil.virtual_memory()
    print(f'RAM    : {ram.total/1e9:.1f} GB total | {ram.available/1e9:.1f} GB free')
    if ram.total < 10e9:
        print('⚠️  RAM < 10GB — giảm BATCH_SHARDS=5 trong notebook 01')
except ImportError:
    pass

print('=' * 55)

In [ ]:
# ── Cell 5: Verify imports ─────────────────────────────────────────────────
errors = []
checks = [
    ('torch',        'import torch; print(f"  ✓ PyTorch {torch.__version__} | CUDA={torch.cuda.is_available()}")', True),
    ('transformers', 'import transformers; print(f"  ✓ transformers {transformers.__version__}")', True),
    ('datasets',     'import datasets; print(f"  ✓ datasets {datasets.__version__}")', True),
    ('accelerate',   'import accelerate; print(f"  ✓ accelerate {accelerate.__version__}")', True),
    ('evaluate',     'import evaluate; print(f"  ✓ evaluate {evaluate.__version__}")', True),
    ('jiwer',        'import jiwer; print(f"  ✓ jiwer")', True),
    ('librosa',      'import librosa; print(f"  ✓ librosa {librosa.__version__}")', True),
    ('soundfile',    'import soundfile; print(f"  ✓ soundfile")', True),
    ('mlflow',       'import mlflow; print(f"  ✓ mlflow {mlflow.__version__}")', True),
    ('pandas',       'import pandas as pd; print(f"  ✓ pandas {pd.__version__}")', True),
    ('numpy',        'import numpy as np; print(f"  ✓ numpy {np.__version__}")', True),
    ('pyarrow',      'import pyarrow; print(f"  ✓ pyarrow {pyarrow.__version__}")', True),
    ('scikit-learn', 'import sklearn; print(f"  ✓ scikit-learn {sklearn.__version__}")', False),
]

print('Verifying imports...')
for name, stmt, required in checks:
    try:
        exec(stmt)
    except Exception as e:
        tag = '❌' if required else '⚠️'
        errors.append(f'{tag} {name}: {e}')
        print(f'  {tag} {name}: {e}')

critical = [e for e in errors if '❌' in e]
if critical:
    print(f'\n❌ {len(critical)} lỗi — chạy lại Cell 3')
else:
    print('\n✓ Tất cả imports OK! Sẵn sàng chạy notebooks.')

In [ ]:
# ── Cell 6: HuggingFace Token + Environment ───────────────────────────────
HF_TOKEN = ''
if IS_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN') or ''
        if HF_TOKEN:
            print('✓ HF_TOKEN loaded from Colab Secrets')
    except Exception:
        pass

if HF_TOKEN:
    os.environ['HF_TOKEN']              = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
else:
    print('ℹ️  HF_TOKEN không có — OK cho public models')

os.environ['MLFLOW_TRACKING_URI']    = os.path.join(LOGS_DIR, 'mlflow')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'

print('✓ Environment configured')

In [ ]:
# ── Cell 7: Pipeline State ─────────────────────────────────────────────────
import json
from pathlib import Path

steps = [
    ('01_dataset_prep', os.path.join(DATASET_EXPORT, 'vietspeech-vimd')),
    ('03_asr_wav2vec2', os.path.join(WEIGHTS_DIR, 'asr_wav2vec2', 'config.json')),
    ('04_translation',  os.path.join(WEIGHTS_DIR, 'translation', 'config.json')),
    ('05_tts',          os.path.join(WEIGHTS_DIR, 'xtts-v2', 'config.json')),
]

print('Pipeline State:')
for step_name, artifact_path in steps:
    done   = os.path.exists(artifact_path)
    status = '✅ Done' if done else '⏳ Pending'
    print(f'  {step_name:30s} {status}')

print('\n✓ Setup complete!')
print('Next: chạy 01_dataset_prep.ipynb')